# ToE remote batch on Colab (inference frontier + GroupKAN)

Generates all remaining experimental data for the AAAI paper_v8 extension.

**Before running:** `Runtime > Change runtime type > GPU` (A100 or L4 if you have
Pro; T4 works but is ~3-5x slower).

**If the runtime disconnects at any point: just `Runtime > Run all` again.**
Every stage is resumable, and all results live on your Google Drive
(`MyDrive/ToE_outputs`), so completed work is never lost or redone.

Expected total GPU time: ~1-3 h on A100, ~2-6 h on L4, ~4-10 h on T4
(free-tier T4 may need 2-3 sessions; that is fine).

In [ ]:
# 1) GPU sanity
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: set Runtime > Change runtime type > GPU'

In [ ]:
# 2) Mount Drive (results persist here across disconnects)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# 3) Clone branch, point outputs/ at Drive, extract checkpoints, install deps
set -e
cd /content
[ -d ToE ] || git clone -b feature/hamiltonian-kan https://github.com/Hafez-Al-Khatib/ToE.git
mkdir -p /content/drive/MyDrive/ToE_outputs
cd ToE
if [ ! -L outputs ]; then rm -rf outputs; ln -s /content/drive/MyDrive/ToE_outputs outputs; fi
tar xzf remote_checkpoints.tar.gz   # pretrained checkpoints -> outputs/ (on Drive)
pip -q install lpips
echo SETUP_OK

In [ ]:
%%bash
# 4) THE BATCH (blocks for hours; live log below).
#    Order: GroupKAN training -> 60-cell frontier sweep (45-min watchdog,
#    resumable parts) -> GroupKAN fine-grid alpha -> LPIPS.
cd /content/ToE
bash scripts/remote_run.sh
cp -f remote_run.log /content/drive/MyDrive/ToE_outputs/ || true

In [ ]:
# 5) Progress check (safe to run anytime, even mid-sweep)
import json, pathlib
parts = pathlib.Path('/content/ToE/outputs/inference_frontier/parts')
done = 0
for p in sorted(parts.glob('*.json')):
    try:
        done += bool(json.loads(p.read_text()).get('done'))
    except Exception:
        pass
print(f'sweep: {done}/60 cells done')
gk = pathlib.Path('/content/ToE/outputs/group_kan')
for f in ['group_kan_8k.pt', 'group_kan_32k.pt',
          'fine_grid_group_kan_8k.json', 'fine_grid_group_kan_32k.json']:
    print(f'{f}:', 'OK' if (gk / f).exists() else 'pending')
fl = pathlib.Path('/content/ToE/outputs/inference_frontier/frontier_lpips.json')
print('frontier_lpips.json:', 'OK' if fl.exists() else 'pending')

In [ ]:
%%bash
# 6) Package results (also kept on Drive as a backup)
cd /content/ToE
tar czhf results_back.tar.gz outputs/inference_frontier outputs/group_kan remote_run.log
cp -f results_back.tar.gz /content/drive/MyDrive/
ls -la results_back.tar.gz

In [ ]:
# 7) Download to your machine (also in Drive root as backup)
from google.colab import files
files.download('/content/ToE/results_back.tar.gz')

## Afterwards, on the Windows machine

Put `results_back.tar.gz` at the repo root, then in PowerShell:

```powershell
tar xzf results_back.tar.gz
```

Then tell Claude: **"remote results are back"** — the pre-registered decision
gate, verdict, and paper edits continue locally from the part files.